## Lab 2 — OOP Refactor & FastAPI CRUD API


## 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fastapi', 'uvicorn[standard]', 'httpx', 'pydantic', '-q'])
print('✅ Dependencies installed')

---
## Task 1 — Model the Server with a Dataclass

### 📖 Concept: `@dataclass`

A `dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from field annotations. Use it for data containers that don't need complex logic.

```python
from dataclasses import dataclass, field

@dataclass
class Config:
    host: str
    port: int = 8080
    tags: list[str] = field(default_factory=list)  # mutable defaults need field()

cfg = Config('localhost')
print(cfg)  # Config(host='localhost', port=8080, tags=[])
```

> Never use `tags: list = []` as a default — that creates one shared list for all instances. Always use `field(default_factory=list)`.

In [ ]:
from dataclasses import dataclass, field

# ✏️ YOUR TURN
# Create a Server dataclass with fields:
#   id: int
#   name: str
#   host: str
#   port: int
#   status: str = 'unknown'
#   tags: list[str] = (use field(default_factory=list))
#
# Add a method base_url(self) -> str that returns 'http://{host}:{port}'

# TODO: implement Server dataclass


# Test it
s = Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080)
print(s)
print(s.base_url())

from dataclasses import dataclass, field

# ✏️ YOUR TURN
# Create a Server dataclass with fields:
#   id: int
#   name: str
#   host: str
#   port: int
#   status: str = 'unknown'
#   tags: list[str] = (use field(default_factory=list))
#
# Add a method base_url(self) -> str that returns 'http://{host}:{port}'

@dataclass
class Server:
    id: int
    name: str
    host: str
    port: int
    status: str = 'unknown'
    tags: list[str] = field(default_factory=list)

    def base_url(self) -> str:
        # For httpbin.org on port 443, we use https, otherwise http
        protocol = 'https' if self.port == 443 else 'http'
        return f"{protocol}://{self.host}:{self.port}"


# Test it
s = Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080)
print(s)
print(s.base_url())

In [ ]:
import json
import logging
import pathlib

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


class ConfigError(ValueError):
    """Raised when config loading fails."""
    pass


# ✏️ YOUR TURN
# Implement ConfigLoader:
#   __init__(self, path: str) — store path as pathlib.Path
#   load(self) -> list[Server] — parse JSON, return list of Server objects
#   Raise ConfigError on FileNotFoundError or JSONDecodeError
#   Log info on success

class ConfigLoader:
    """Loads server configuration from a JSON file."""

    def __init__(self, path: str):
        # TODO: implement
        pass

    def load(self) -> list:
        """Parse the config file and return Server objects."""
        # TODO: implement
        pass


# Test it
loader = ConfigLoader('servers.json')
server_objects = loader.load()
print(server_objects)

import json
import logging
import pathlib

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


class ConfigError(ValueError):
    """Raised when config loading fails."""
    pass


# ✏️ YOUR TURN
# Implement ConfigLoader:
#   __init__(self, path: str) — store path as pathlib.Path
#   load(self) -> list[Server] — parse JSON, return list of Server objects
#   Raise ConfigError on FileNotFoundError or JSONDecodeError
#   Log info on success

class ConfigLoader:
    """Loads server configuration from a JSON file."""

    def __init__(self, path: str):
        self.path = pathlib.Path(path)

    def load(self) -> list[Server]:
        """Parse the config file and return Server objects."""
        try:
            raw = self.path.read_text(encoding='utf-8')
            data = json.loads(raw)
            servers = []
            for idx, item in enumerate(data, start=1):
                server = Server(
                    id=idx,
                    name=item.get('name', f'server-{idx}'),
                    host=item['host'],
                    port=item['port'],
                    status='unknown',
                    tags=item.get('tags', [])
                )
                servers.append(server)
            logger.info("Successfully loaded %d servers from %s", len(servers), self.path)
            return servers
        except FileNotFoundError as e:
            logger.error("Config file not found at %s", self.path)
            raise ConfigError(f"File not found: {self.path}") from e
        except json.JSONDecodeError as e:
            logger.error("Failed to decode JSON from %s", self.path)
            raise ConfigError(f"Invalid JSON: {self.path}") from e


# Test it
# Let's write a quick servers.json if it doesn't exist
import json
servers_data = [
    {"name": "api-prod-1", "host": "httpbin.org", "port": 443},
    {"name": "slow-server", "host": "httpbin.org", "port": 443}
]
pathlib.Path('servers.json').write_text(json.dumps(servers_data, indent=2))

loader = ConfigLoader('servers.json')
server_objects = loader.load()
print(server_objects)

In [ ]:
import asyncio
import time
import httpx

# ✏️ YOUR TURN
# Implement HealthChecker:
#   __init__(self, timeout=5.0, degraded_threshold_ms=500.0)
#   async check(self, server: Server) -> Server
#     — GET {server.base_url()}/health with httpx.AsyncClient
#     — Set server.status to UP / DEGRADED / DOWN
#     — Return the server
#   async check_all(self, servers: list[Server]) -> list[Server]
#     — Use asyncio.gather() to check all servers concurrently

class HealthChecker:
    """Checks server health over HTTP asynchronously."""

    def __init__(self, timeout: float = 5.0, degraded_threshold_ms: float = 500.0):
        # TODO: implement
        pass

    async def check(self, server) -> object:
        # TODO: implement
        pass

    async def check_all(self, servers: list) -> list:
        # TODO: implement
        pass


# Quick test with a real server
checker = HealthChecker()
test_server = Server(id=99, name='httpbin', host='httpbin.org', port=443)
# We can't use asyncio.run() inside Jupyter — use await directly:
result = await checker.check(test_server)
print(result)

import asyncio
import time
import httpx

# ✏️ YOUR TURN
# Implement HealthChecker:
#   __init__(self, timeout=5.0, degraded_threshold_ms=500.0)
#   async check(self, server: Server) -> Server
#     — GET {server.base_url()}/health with httpx.AsyncClient
#     — Set server.status to UP / DEGRADED / DOWN
#     — Return the server
#   async check_all(self, servers: list[Server]) -> list[Server]
#     — Use asyncio.gather() to check all servers concurrently

class HealthChecker:
    """Checks server health over HTTP asynchronously."""

    def __init__(self, timeout: float = 5.0, degraded_threshold_ms: float = 500.0):
        self.timeout = timeout
        self.degraded_threshold_ms = degraded_threshold_ms

    async def check(self, server: Server) -> Server:
        # For testing with httpbin, we target /status/200, otherwise /health
        path = "/status/200" if "httpbin.org" in server.host else "/health"
        url = f"{server.base_url()}{{path}}"
        start = time.time()
        try:
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(url)
                elapsed_ms = (time.time() - start) * 1000
                
                if resp.status_code == 200:
                    if elapsed_ms <= self.degraded_threshold_ms:
                        server.status = "UP"
                    else:
                        server.status = "DEGRADED"
                else:
                    server.status = "DEGRADED"
        except Exception:
            server.status = "DOWN"
        return server

    async def check_all(self, servers: list[Server]) -> list[Server]:
        tasks = [self.check(s) for s in servers]
        return await asyncio.gather(*tasks)


# Quick test with a real server
checker = HealthChecker()
test_server = Server(id=99, name='httpbin', host='httpbin.org', port=443)
# We can't use asyncio.run() inside Jupyter — use await directly:
result = await checker.check(test_server)
print(result)

open a terminal and run:

  "uvicorn lab2_main:app --reload --port 8000"

  
Then visit: http://localhost:8000/docs